# 🧠 Advanced Retrieval-Augmented Generation (RAG) with LangChain
### Building a Multi-Source AI Research Assistant using Azure OpenAI

---

## 🎯 Learning Objectives

This notebook builds a **production-grade Retrieval-Augmented Generation (RAG)** system capable of synthesizing information from multiple real-world sources. By the end, you will be able to:

1. Understand why LLMs hallucinate and how RAG solves this problem at its root.
2. Build a complete RAG pipeline using LangChain's modern **LCEL** (LangChain Expression Language).
3. Ingest and query Wikipedia articles using semantic vector search.
4. **[Advanced]** Programmatically search arXiv, download real scientific PDFs, and build a multi-paper knowledge base.
5. Engineer prompts that force the model to cite sources and prevent hallucination.
6. Evaluate retrieval quality using the **LLM-as-a-Judge** pattern.

---

## 🏗️ The RAG Architecture

```text
╔══════════════════════════════════════════════════════════════════════╗
║                     PHASE 1: INGESTION (Offline)                    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  [Raw Documents]  ──►  [Text Splitter]  ──►  [Embedding Model]      ║
║  (PDFs, Wikipedia)      (Chunks ~1000       (Azure text-embedding)  ║
║                          chars each)                                 ║
║                                                 │                    ║
║                                                 ▼                    ║
║                                          [FAISS Vector DB]           ║
║                                          (Dense Index)               ║
╠══════════════════════════════════════════════════════════════════════╣
║                  PHASE 2: RETRIEVAL & GENERATION (Runtime)          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  [User Question]  ──►  [Embed Question]  ──►  [Similarity Search]   ║
║                                                       │              ║
║                                                       ▼              ║
║                                             [Top-K Relevant Chunks]  ║
║                                                       │              ║
║                                                       ▼              ║
║                                          [Prompt Template + Context] ║
║                                                       │              ║
║                                                       ▼              ║
║                                             [Azure OpenAI GPT-4.1]  ║
║                                                       │              ║
║                                                       ▼              ║
║                                        [Grounded Answer + Citations] ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📋 Notebook Structure

| Section | Topic | Complexity |
|---------|-------|------------|
| 1 | Setup & Azure OpenAI Configuration | ⭐ |
| 2 | Baseline: The Hallucination Problem | ⭐⭐ |
| 3 | Level 1 RAG: Wikipedia Knowledge Base | ⭐⭐⭐ |
| 4 | Level 2 RAG: Multi-Paper arXiv Research Assistant | ⭐⭐⭐⭐ |
| 5 | Advanced: Citation-Enforced Synthesis | ⭐⭐⭐⭐ |
| 6 | Advanced: Cross-Source Comparison Query | ⭐⭐⭐⭐⭐ |
| 7 | Evaluation: LLM-as-a-Judge for Retrieval Quality | ⭐⭐⭐⭐⭐ |


---
## ⚙️ Section 1: Setup and Azure OpenAI Configuration

We use the modern LangChain ecosystem (v0.3+), which uses **LCEL (LangChain Expression Language)** — a declarative, composable way to build chains using the `|` operator. This is the current recommended approach and replaces the older `RetrievalQA` and `LLMChain` classes.

### Key Components

| Component | Library | Role |
|-----------|---------|------|
| `AzureChatOpenAI` | `langchain-openai` | LLM for text generation |
| `AzureOpenAIEmbeddings` | `langchain-openai` | Converts text to vectors |
| `FAISS` | `langchain-community` | Vector similarity search |
| `WikipediaLoader` | `langchain-community` | Loads Wikipedia articles |
| `PyPDFLoader` | `langchain-community` | Parses PDF files |
| `RecursiveCharacterTextSplitter` | `langchain-text-splitters` | Chunks documents |


In [1]:
# Install required libraries
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters
!pip install -q faiss-cpu wikipedia arxiv pypdf requests
print("✅ All libraries installed.")



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ All libraries installed.


In [2]:
import os
from dotenv import load_dotenv

# ── Remove any conflicting base URL environment variables ─────────────────────
# These can interfere with Azure OpenAI routing in some environments
for _k in ["OPENAI_BASE_URL", "OPENAI_API_BASE"]:
    os.environ.pop(_k, None)

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# ── LLM: GPT-4.1 for generation ───────────────────────────────────────────────
llm = AzureChatOpenAI(
    azure_deployment="gpt-4.1",
    temperature=0.0,   # Deterministic for factual retrieval
    max_tokens=1000
)

# ── Embeddings: Azure text-embedding for semantic search ──────────────────────
embeddings = AzureOpenAIEmbeddings(
    azure_deployment="textembedding-test-exquitech"
)

# Quick sanity check
test_vec = embeddings.embed_query("Hello, RAG!")
print("Azure OpenAI LLM and Embeddings initialized!")
print(f"   LLM Deployment:       gpt-4.1")
print(f"   Embedding Deployment: textembedding-test-exquitech")
print(f"   Embedding Dimensions: {len(test_vec)}")

Azure OpenAI LLM and Embeddings initialized!
   LLM Deployment:       gpt-4.1
   Embedding Deployment: textembedding-test-exquitech
   Embedding Dimensions: 1536


---
## 2️⃣ Baseline: The Hallucination Problem (Without RAG)

Before building our RAG pipeline, we must understand **why it is necessary**. Large language models store knowledge in their weights during training — this is called **parametric memory**. This memory has two critical limitations:

1. **Knowledge Cutoff**: The model has no knowledge of events after its training cutoff date.
2. **Hallucination**: When asked about specific, obscure, or proprietary information, the model invents plausible-sounding but false answers.

### Why Does Hallucination Happen?

```text
Training: [Billions of text tokens] ──► [Compressed into ~billions of weights]
                                                    │
                                                    ▼
                                    Lossy compression: details are lost!
                                    The model "fills in gaps" with plausible text.
```

We will demonstrate this by asking about a completely **fictional AI paper**. Watch what happens.


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── A question about a completely fictional paper ─────────────────────────────
hallucination_question = (
    "Explain the core mechanism of the Quantum-Entangled Transformer Network (QETN) "
    "published by Dr. Elara Vance and Dr. Julian Vance in 2026. "
    "Describe its architecture and how it differs from standard transformers."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI research scientist. Answer the user's question directly and confidently."),
    ("human", "{question}")
])

# Build the chain using LCEL (the | operator)
# This is the modern LangChain way: prompt | llm | parser
chain = prompt | llm | StrOutputParser()

print("❓ Question (about a FICTIONAL paper):")
print(f"   {hallucination_question}\n")
print("🤖 Model Response (Without RAG):")
print("-" * 60)
response = chain.invoke({"question": hallucination_question})
print(response)
print("-" * 60)
print("\n⚠️  Notice: The model may fabricate details about a paper that does not exist.")
print("   This is the hallucination problem that RAG solves by grounding answers in real documents.")


❓ Question (about a FICTIONAL paper):
   Explain the core mechanism of the Quantum-Entangled Transformer Network (QETN) published by Dr. Elara Vance and Dr. Julian Vance in 2026. Describe its architecture and how it differs from standard transformers.

🤖 Model Response (Without RAG):
------------------------------------------------------------
Certainly! The **Quantum-Entangled Transformer Network (QETN)**, as introduced by Dr. Elara Vance and Dr. Julian Vance in 2026, is a novel neural architecture that integrates principles of quantum entanglement into the transformer framework to enhance information representation and contextual reasoning.

---

### Core Mechanism

The QETN’s core mechanism is the **Quantum Entanglement Attention (QEA)** module. Unlike standard self-attention, which computes pairwise relationships between tokens, QEA models *multi-way correlations* inspired by quantum entanglement. This allows the network to capture complex, non-local dependencies that are difficult

---
## 3️⃣ Level 1 RAG: Wikipedia Knowledge Base

Now we implement a full RAG pipeline. The key insight is that we are replacing the model's **parametric memory** with **non-parametric, external memory** stored in a vector database.

### The Chunking Strategy

A critical design decision in RAG is how to split documents. The `RecursiveCharacterTextSplitter` tries to split on natural boundaries (paragraphs, sentences) before resorting to arbitrary character splits.

```text
Document: [────────────────────────────────────────────────────────────]
Chunk 1:  [═══════════════════════]
Chunk 2:              [═══════════════════════]  ← 200 char overlap
Chunk 3:                          [═══════════════════════]
```

**Why overlap?** If a key sentence falls at the boundary between two chunks, the overlap ensures it appears in at least one complete chunk.

### The LCEL RAG Chain

The modern LangChain way to build a RAG chain is:

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
```

This is a **declarative pipeline**: the `|` operator connects components, and data flows left to right.


In [4]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough

# ── Step 1: Load Wikipedia content ────────────────────────────────────────────
print("📚 Fetching Wikipedia article on 'Retrieval-augmented generation'...")
loader = WikipediaLoader(
    query="Retrieval-augmented generation",
    lang="en",
    load_max_docs=1,
    doc_content_chars_max=20000
)
wiki_docs = loader.load()
print(f"   Loaded: {len(wiki_docs)} document | {len(wiki_docs[0].page_content):,} characters")
print(f"   Source: {wiki_docs[0].metadata.get('source', 'Wikipedia')}")

# ── Step 2: Split into overlapping chunks ─────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " "]
)
wiki_chunks = text_splitter.split_documents(wiki_docs)
avg_len = sum(len(c.page_content) for c in wiki_chunks) // len(wiki_chunks)
print(f"\n✂️  Chunks created: {len(wiki_chunks)} (avg {avg_len} chars each)")

# ── Step 3: Embed and store in FAISS ──────────────────────────────────────────
print("\n🧠 Generating embeddings and building FAISS vector index...")
wiki_vectorstore = FAISS.from_documents(wiki_chunks, embeddings)
wiki_retriever = wiki_vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Vector database ready!")
print("   Each chunk is now a point in 1536-dimensional embedding space.")
print("   Similarity search finds the nearest neighbors to any query vector.")


📚 Fetching Wikipedia article on 'Retrieval-augmented generation'...
   Loaded: 1 document | 10,842 characters
   Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation

✂️  Chunks created: 17 (avg 654 chars each)

🧠 Generating embeddings and building FAISS vector index...
✅ Vector database ready!
   Each chunk is now a point in 1536-dimensional embedding space.
   Similarity search finds the nearest neighbors to any query vector.


In [5]:
# ── Helper: format retrieved documents for the prompt ─────────────────────────
def format_docs(docs):
    """Concatenate retrieved chunks into a single context string with source labels."""
    return "\n\n---\n\n".join(
        f"[Source: {doc.metadata.get('title', 'Wikipedia')}]\n{doc.page_content}"
        for doc in docs
    )

# ── RAG Prompt: strictly grounded ─────────────────────────────────────────────
wiki_rag_prompt = ChatPromptTemplate.from_template("""
You are a specialized AI assistant. Answer the question based ONLY on the provided context.
If the context does not contain the answer, say "I cannot answer this based on the provided context."
Do NOT use your internal training knowledge.

Context:
{context}

Question: {question}
Answer:""")

# ── Build the LCEL RAG chain ───────────────────────────────────────────────────
wiki_rag_chain = (
    {"context": wiki_retriever | format_docs, "question": RunnablePassthrough()}
    | wiki_rag_prompt
    | llm
    | StrOutputParser()
)

# ── Test the chain ─────────────────────────────────────────────────────────────
questions = [
    "What are the main advantages of RAG over fine-tuning a model?",
    "What is the role of the retriever in a RAG system?",
    "What types of knowledge sources can RAG systems use?",
]

for q in questions:
    print(f"❓ {q}")
    answer = wiki_rag_chain.invoke(q)
    print(f"🤖 {answer}")
    print()


❓ What are the main advantages of RAG over fine-tuning a model?
🤖 The main advantages of RAG over fine-tuning a model are:

1. RAG reduces the need to retrain LLMs with new data, saving on computational and financial costs.
2. When new information becomes available, it is sufficient to augment the model’s external knowledge base with the updated information, rather than retraining the model.
3. RAG allows LLMs to include sources in their responses, enabling users to verify the cited sources and providing greater transparency.
4. RAG helps reduce AI hallucinations by allowing the model to pull relevant text from databases or documents, helping LLMs stick to the facts.

❓ What is the role of the retriever in a RAG system?
🤖 I cannot answer this based on the provided context.

❓ What types of knowledge sources can RAG systems use?
🤖 RAG systems can use databases, uploaded documents, or web sources as knowledge sources.



In [6]:
# ── Inspect what the retriever actually fetched ───────────────────────────────
print("🔍 Transparency: What chunks were retrieved for the last question?\n")
retrieved_docs = wiki_retriever.invoke(questions[-1])
for i, doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1} ({len(doc.page_content)} chars):")
    print(f"  {doc.page_content[:250]}...")
    print()
print("💡 These are the exact passages the model used to generate its answer.")
print("   No hallucination is possible when the model is constrained to this context.")


🔍 Transparency: What chunks were retrieved for the last question?

Chunk 1 (702 chars):
  === RAG poisoning ===
RAG systems may retrieve factually correct but misleading sources, leading to errors in interpretation. In some cases, an LLM may extract statements from a source without considering its context, resulting in an incorrect conclu...

Chunk 2 (984 chars):
  RAG improves large language models (LLMs) by incorporating information retrieval before generating responses. Unlike LLMs that rely on static training data, RAG pulls relevant text from databases, uploaded documents, or web sources. According to Ars ...

Chunk 3 (310 chars):
  === Evaluation and benchmarks ===
RAG systems are commonly evaluated using benchmarks designed to test retrievability, retrieval accuracy and generative quality. Popular datasets include BEIR, a suite of information retrieval tasks across diverse dom...

💡 These are the exact passages the model used to generate its answer.
   No hallucination is possib

---
## 4️⃣ Level 2 RAG: Multi-Paper arXiv Research Assistant

This is where the real challenge begins. We will build a system that can answer questions by synthesizing information across **multiple scientific papers** on a specific AI topic.

### Research Topic: Agentic LLM Workflows

We will search arXiv for recent papers on **"Agentic LLM Workflows"** — a cutting-edge topic covering how LLMs can autonomously plan and execute multi-step tasks. This topic is recent enough that the model's parametric memory may be incomplete or outdated.

### Why Multi-Paper RAG Is Hard

```text
Paper 1: Uses term "tool use"
Paper 2: Uses term "function calling"     ← Same concept, different vocabulary
Paper 3: Uses term "action execution"

The retriever must match queries to the right chunks
despite vocabulary mismatches — this is why dense
semantic embeddings outperform keyword search here.
```

### Download Strategy

We use the `arxiv` Python library to search and get metadata, then download PDFs using `requests` with proper headers to avoid rate limiting.


In [7]:
import arxiv
import requests
import os
import time
import glob

# ── Configuration ─────────────────────────────────────────────────────────────────
SEARCH_QUERY = "Agentic LLM workflows planning reasoning"
MAX_PAPERS = 5
DOWNLOAD_DIR = "./arxiv_papers"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# ── Check if papers are already downloaded ──────────────────────────────────────
existing_pdfs = sorted(glob.glob(os.path.join(DOWNLOAD_DIR, "*.pdf")))

if len(existing_pdfs) >= MAX_PAPERS:
    print(f"Papers already downloaded ({len(existing_pdfs)} found). Skipping arXiv search.")
    pdf_paths = existing_pdfs[:MAX_PAPERS]
    paper_metadata = {}
    for fpath in pdf_paths:
        fname = os.path.basename(fpath)
        paper_metadata[fpath] = {
            "title": fname.replace(".pdf", "").replace("_", " "),
            "authors": "See paper",
            "published": "2025",
            "arxiv_id": fname.split("_")[1] if "_" in fname else "unknown",
            "abstract": "Previously downloaded paper."
        }
        print(f"   Using: {fname}")
else:
    print(f"Searching arXiv for: '{SEARCH_QUERY}'")
    print(f"   Target: {MAX_PAPERS} papers\n")

    # ── Search arXiv ──────────────────────────────────────────────────────────────────
    arxiv_client = arxiv.Client()
    search = arxiv.Search(
        query=SEARCH_QUERY,
        max_results=MAX_PAPERS,
        sort_by=arxiv.SortCriterion.Relevance
    )

    results = list(arxiv_client.results(search))
    print(f"Found {len(results)} papers. Downloading PDFs...\n")

    # ── Download PDFs using requests ────────────────────────────────────────────────
    headers = {"User-Agent": "Mozilla/5.0 (compatible; research-notebook/1.0)"}
    pdf_paths = []
    paper_metadata = {}

    for i, result in enumerate(results):
        safe_title = "".join([c if c.isalnum() else "_" for c in result.title])[:50]
        fname = f"paper_{i+1:02d}_{safe_title}.pdf"
        fpath = os.path.join(DOWNLOAD_DIR, fname)

        print(f"[{i+1}/{MAX_PAPERS}] {result.title[:65]}...")
        print(f"        Authors: {', '.join([a.name for a in result.authors[:3]])}{'...' if len(result.authors) > 3 else ''}")
        print(f"        Published: {result.published.date()} | arXiv ID: {result.entry_id.split('/')[-1]}")

        try:
            resp = requests.get(result.pdf_url, headers=headers, timeout=60)
            resp.raise_for_status()
            with open(fpath, "wb") as f:
                f.write(resp.content)
            size_kb = os.path.getsize(fpath) // 1024
            print(f"        Saved: {size_kb} KB")

            pdf_paths.append(fpath)
            paper_metadata[fpath] = {
                "title": result.title,
                "authors": ", ".join([a.name for a in result.authors]),
                "published": str(result.published.date()),
                "arxiv_id": result.entry_id.split("/")[-1],
                "abstract": result.summary[:300] + "..."
            }
        except Exception as e:
            print(f"        Failed: {e}")

        time.sleep(1)
        print()

print(f"\nReady: {len(pdf_paths)} papers available for processing.")


Papers already downloaded (5 found). Skipping arXiv search.
   Using: paper_01_Agentic_Reasoning_for_Large_Language_Models.pdf
   Using: paper_02_Don_t_Vibe_Code__Do_Skele_Code__Interactive_No_Cod.pdf
   Using: paper_03_When_Reasoning_Beats_Scale__A_1_5B_Reasoning_Model.pdf
   Using: paper_04_Navigating_the_State_of_Cognitive_Flow__Context_Aw.pdf
   Using: paper_05_PROV_AGENT__Unified_Provenance_for_Tracking_AI_Age.pdf

Ready: 5 papers available for processing.


### 📄 Parsing PDFs and Injecting Metadata

After downloading, we parse each PDF and **inject rich metadata** into every chunk. This metadata is crucial for:
1. **Attribution**: Knowing which paper a chunk came from.
2. **Citation generation**: Providing title and page number for citations.
3. **Filtering**: Allowing future queries to restrict to specific papers.


In [8]:
from langchain_community.document_loaders import PyPDFLoader

# ── Parse PDFs and inject rich metadata ───────────────────────────────────────
all_paper_chunks = []

print("📄 Parsing and chunking all papers...\n")

for path in pdf_paths:
    meta = paper_metadata[path]
    print(f"Processing: {meta['title'][:60]}...")
    
    # Load the PDF (PyPDFLoader adds 'page' to metadata automatically)
    loader = PyPDFLoader(path)
    pages = loader.load()
    
    # Inject our rich metadata into every page document
    for page in pages:
        page.metadata["source_title"] = meta["title"]
        page.metadata["authors"] = meta["authors"]
        page.metadata["published"] = meta["published"]
        page.metadata["arxiv_id"] = meta["arxiv_id"]
    
    # Split into chunks
    chunks = text_splitter.split_documents(pages)
    all_paper_chunks.extend(chunks)
    
    print(f"  → {len(pages)} pages → {len(chunks)} chunks")

print(f"\n{'='*60}")
print(f"Total chunks across all {len(pdf_paths)} papers: {len(all_paper_chunks)}")
print(f"{'='*60}")
print(f"\nSample chunk metadata:")
sample = all_paper_chunks[5]
print(f"  Title:    {sample.metadata['source_title'][:50]}...")
print(f"  Authors:  {sample.metadata['authors'][:50]}...")
print(f"  Page:     {sample.metadata.get('page', 0) + 1}")
print(f"  arXiv ID: {sample.metadata['arxiv_id']}")


📄 Parsing and chunking all papers...

Processing: paper 01 Agentic Reasoning for Large Language Models...
  → 135 pages → 596 chunks
Processing: paper 02 Don t Vibe Code  Do Skele Code  Interactive No Cod...
  → 26 pages → 84 chunks
Processing: paper 03 When Reasoning Beats Scale  A 1 5B Reasoning Model...
  → 20 pages → 89 chunks
Processing: paper 04 Navigating the State of Cognitive Flow  Context Aw...
  → 4 pages → 18 chunks
Processing: paper 05 PROV AGENT  Unified Provenance for Tracking AI Age...
  → 7 pages → 47 chunks

Total chunks across all 5 papers: 834

Sample chunk metadata:
  Title:    paper 01 Agentic Reasoning for Large Language Mode...
  Authors:  See paper...
  Page:     2
  arXiv ID: 01


In [9]:
# ── Build the Multi-Paper Vector Database ─────────────────────────────────────
print("🧠 Building unified FAISS index for all papers...")
print(f"   Embedding {len(all_paper_chunks)} chunks into 1536-dimensional space...")
print("   (This may take a minute — we're making one API call per chunk)\n")

arxiv_vectorstore = FAISS.from_documents(all_paper_chunks, embeddings)

# For scientific synthesis, retrieve more chunks for broader coverage
arxiv_retriever = arxiv_vectorstore.as_retriever(search_kwargs={"k": 6})

print("✅ Multi-paper vector database ready!")
print(f"   Total vectors indexed: {len(all_paper_chunks)}")
print(f"   Retrieval k=6: each query returns top 6 most similar chunks")


🧠 Building unified FAISS index for all papers...
   Embedding 834 chunks into 1536-dimensional space...
   (This may take a minute — we're making one API call per chunk)

✅ Multi-paper vector database ready!
   Total vectors indexed: 834
   Retrieval k=6: each query returns top 6 most similar chunks


---
## 5️⃣ Advanced: Citation-Enforced Synthesis

In academic and professional settings, it is not enough to get the right answer — you must be able to **trace every claim back to its source**. We will engineer a prompt that:

1. Forces the model to cite the paper title and page number for every claim.
2. Explicitly prohibits the model from using its internal knowledge.
3. Requires the model to acknowledge when the papers do not address a question.

### The Citation Format

We instruct the model to use: `[Title, p.X]`

For example: *"The paper describes a planning module that decomposes goals into subtasks [Agentic Reasoning for Large Language Models, p.3]."*

This is a critical safety mechanism for high-stakes applications like medical, legal, or financial research assistants.


In [10]:
# ── Citation-aware document formatter ────────────────────────────────────────
def format_docs_with_citations(docs):
    """Format retrieved chunks with full citation metadata for the LLM to reference."""
    formatted = []
    for i, doc in enumerate(docs):
        title = doc.metadata.get("source_title", "Unknown")
        authors = doc.metadata.get("authors", "Unknown")
        page = doc.metadata.get("page", 0)
        arxiv_id = doc.metadata.get("arxiv_id", "")
        
        block = (
            f"[DOCUMENT {i+1}]\n"
            f"Title: {title}\n"
            f"Authors: {authors}\n"
            f"Page: {page + 1}\n"
            f"arXiv ID: {arxiv_id}\n"
            f"Content:\n{doc.page_content}"
        )
        formatted.append(block)
    return ("\n\n" + "="*60 + "\n\n").join(formatted)

# ── Citation-enforced prompt ───────────────────────────────────────────────────
citation_prompt = ChatPromptTemplate.from_template("""
You are an expert AI research assistant tasked with synthesizing scientific literature.

STRICT RULES:
1. Answer ONLY using information from the provided document blocks below.
2. For EVERY factual claim, you MUST cite the source using this format: [Title, p.X]
3. If the documents do not address the question, state: "The provided papers do not address this question."
4. Do NOT use your internal training knowledge under any circumstances.
5. If multiple papers discuss the same topic, synthesize their views and cite each one.

Document Blocks:
{context}

Research Question: {question}

Detailed Answer with Citations:""")

# ── Build the citation-enforced RAG chain ─────────────────────────────────────
citation_rag_chain = (
    {"context": arxiv_retriever | format_docs_with_citations, "question": RunnablePassthrough()}
    | citation_prompt
    | llm
    | StrOutputParser()
)

# ── Query 1: Core architecture question ───────────────────────────────────────
q1 = "What are the key components of agentic LLM workflows and how do they enable autonomous task execution?"
print(f"❓ Research Question 1:")
print(f"   {q1}\n")
print("⏳ Synthesizing from papers...\n")
print("🤖 Answer with Citations:")
print("=" * 70)
print(citation_rag_chain.invoke(q1))


❓ Research Question 1:
   What are the key components of agentic LLM workflows and how do they enable autonomous task execution?

⏳ Synthesizing from papers...

🤖 Answer with Citations:
The key components of agentic LLM workflows include:

1. **Perception, Reasoning, Planning, and Action Grounding**: At the foundational level, autonomous agents must be able to perceive their environment, reason about goals, plan actions, and ground those plans into tool-augmented workflows. This progression enables agents to move from simple evidence retrieval to orchestrating full scientific workflows end-to-end, supporting tasks such as hypothesis generation, data synthesis, and paper writing [paper 01 Agentic Reasoning for Large Language Models, p.57].

2. **Dynamic Tool Use and Action Loops**: Agentic LLMs interleave reasoning steps with tool calls, creating a dynamic loop that allows the agent to interact with external environments. This enables the agent to gather information and execute tasks be

In [11]:
# ── Query 2: Challenges and limitations ──────────────────────────────────────
q2 = "What are the main challenges and failure modes of agentic LLM systems discussed in these papers?"
print(f"❓ Research Question 2:")
print(f"   {q2}\n")
print("⏳ Synthesizing from papers...\n")
print("🤖 Answer with Citations:")
print("=" * 70)
print(citation_rag_chain.invoke(q2))


❓ Research Question 2:
   What are the main challenges and failure modes of agentic LLM systems discussed in these papers?

⏳ Synthesizing from papers...

🤖 Answer with Citations:
The main challenges and failure modes of agentic LLM systems discussed in these papers include:

1. Long-Horizon Planning and Persistent Memory: Agentic LLM systems face difficulties in long-horizon planning, where actions and decisions must be coordinated over extended timeframes. Failures often arise from complex interactions across time and between system components, making it challenging to attribute errors and audit system behavior [paper 01 Agentic Reasoning for Large Language Models, p.74].

2. Inadequate Benchmarks and Guardrails: Existing evaluation benchmarks and safety guardrails are primarily focused on short-horizon behaviors. This leaves planning-time failures and issues arising from multi-agent dynamics underexplored and insufficiently addressed [paper 01 Agentic Reasoning for Large Language Mo

In [12]:
# ── Query 3: Out-of-scope boundary test ──────────────────────────────────────
q3 = "What is the best deep learning architecture for image classification on ImageNet?"
print(f"❓ Research Question 3 (Out-of-scope boundary test):")
print(f"   {q3}\n")
print("⏳ Checking if model respects the context boundary...\n")
print("🤖 Answer:")
print("=" * 70)
print(citation_rag_chain.invoke(q3))
print()
print("💡 The model should say it cannot answer from the provided papers.")
print("   This demonstrates that the prompt engineering is working correctly.")


❓ Research Question 3 (Out-of-scope boundary test):
   What is the best deep learning architecture for image classification on ImageNet?

⏳ Checking if model respects the context boundary...

🤖 Answer:
The provided papers do not address this question.

💡 The model should say it cannot answer from the provided papers.
   This demonstrates that the prompt engineering is working correctly.


---
## 6️⃣ Advanced: Cross-Source Comparison Query

One of the most powerful capabilities of a multi-source RAG system is the ability to **compare and contrast** information across different documents. We will now merge our Wikipedia knowledge base with our arXiv paper index into a single unified store, then ask a question that requires drawing from both sources.

### Why This Is Challenging

```text
Wikipedia: General, encyclopedic definition of RAG
arXiv Papers: Specific, technical implementations of RAG

A cross-source query requires the model to:
1. Retrieve relevant chunks from BOTH sources
2. Identify similarities and differences
3. Synthesize a coherent comparative answer
4. Cite each source correctly
```


In [13]:
# ── Prepare Wikipedia chunks with metadata ────────────────────────────────────
# Add source metadata to distinguish from arXiv papers
for doc in wiki_chunks:
    doc.metadata["source_title"] = "Wikipedia: Retrieval-Augmented Generation"
    doc.metadata["authors"] = "Wikipedia Contributors"
    doc.metadata["published"] = "2024"
    doc.metadata["arxiv_id"] = "N/A (Wikipedia)"

# ── Merge Wikipedia and arXiv into a unified vector store ────────────────────
print("🔗 Merging Wikipedia and arXiv vector stores...")
unified_vectorstore = FAISS.from_documents(all_paper_chunks + wiki_chunks, embeddings)
unified_retriever = unified_vectorstore.as_retriever(search_kwargs={"k": 7})

print(f"✅ Unified vector store created!")
print(f"   arXiv chunks:     {len(all_paper_chunks)}")
print(f"   Wikipedia chunks: {len(wiki_chunks)}")
print(f"   Total vectors:    {len(all_paper_chunks) + len(wiki_chunks)}")


🔗 Merging Wikipedia and arXiv vector stores...
✅ Unified vector store created!
   arXiv chunks:     834
   Wikipedia chunks: 17
   Total vectors:    851


In [14]:
# ── Cross-source comparison chain ─────────────────────────────────────────────
cross_source_prompt = ChatPromptTemplate.from_template("""
You are an expert AI research assistant synthesizing information from multiple sources.

INSTRUCTIONS:
1. Use ONLY the provided document blocks.
2. When information comes from different sources, explicitly compare and contrast them.
3. Cite every claim: [Title, p.X] or [Wikipedia: Title].
4. Organize your answer with clear headings if the question is multi-part.
5. Highlight any contradictions or complementary perspectives between sources.

Document Blocks:
{context}

Question: {question}
Comparative Analysis:""")

cross_source_chain = (
    {"context": unified_retriever | format_docs_with_citations, "question": RunnablePassthrough()}
    | cross_source_prompt
    | llm
    | StrOutputParser()
)

# ── Cross-source query ────────────────────────────────────────────────────────
cross_q = (
    "How do the scientific papers define and implement RAG differently from "
    "the general definition provided in the Wikipedia article? "
    "What aspects do the papers emphasize that Wikipedia does not cover?"
)
print(f"❓ Cross-Source Question:")
print(f"   {cross_q}\n")
print("⏳ Comparing sources...\n")
print("🤖 Comparative Analysis:")
print("=" * 70)
print(cross_source_chain.invoke(cross_q))


❓ Cross-Source Question:
   How do the scientific papers define and implement RAG differently from the general definition provided in the Wikipedia article? What aspects do the papers emphasize that Wikipedia does not cover?

⏳ Comparing sources...

🤖 Comparative Analysis:
## Comparative Analysis: Scientific Papers vs. Wikipedia on RAG

### 1. General Definition of RAG (Wikipedia)

The Wikipedia article defines Retrieval-Augmented Generation (RAG) as a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. The process involves LLMs first retrieving relevant documents and then generating responses based on both the retrieved content and their pre-existing training data. This approach allows LLMs to access domain-specific or updated information, improving factual accuracy and transparency, and reducing hallucinations. RAG is also described as a way to avoid costly retraining and to provide verifiable sources for user qu

---
## 7️⃣ Evaluation: LLM-as-a-Judge for Retrieval Quality

A RAG system is only as good as its retriever. If the retriever fetches irrelevant chunks, the LLM will either hallucinate or say "I don't know." We need a way to **measure retrieval quality** without manually labeling thousands of examples.

### The LLM-as-a-Judge Pattern

We use a second LLM call to evaluate whether each retrieved chunk is actually relevant to the question. This is a standard technique in production RAG systems.

```text
[User Question] ──► [Retriever] ──► [Retrieved Chunks]
                                           │
                                           ▼
                                    [Judge LLM]
                                    "Score each chunk 1-5
                                     for relevance"
                                           │
                                           ▼
                                    [Relevance Scores]
                                    → Diagnose retrieval quality
                                    → Compare configurations
                                    → Identify failure modes
```

### Scoring Rubric

| Score | Meaning |
|-------|---------|
| 5 | Directly answers the question |
| 4 | Highly relevant, provides important context |
| 3 | Somewhat relevant, tangentially related |
| 2 | Barely relevant, only loosely connected |
| 1 | Not relevant at all |


In [15]:
import json

# ── LLM-as-a-Judge prompt ─────────────────────────────────────────────────────
judge_prompt = ChatPromptTemplate.from_template("""
You are an expert evaluator assessing the relevance of retrieved text chunks for a given question.

Rate the relevance of the chunk on a scale of 1-5:
- 5: Directly answers the question
- 4: Highly relevant, provides important context
- 3: Somewhat relevant, tangentially related
- 2: Barely relevant, only loosely connected
- 1: Not relevant at all

Question: {question}

Retrieved Chunk:
{chunk}

Respond with ONLY a JSON object (no other text):
{{"score": <1-5>, "reason": "<one sentence explaining the score>"}}
JSON:""")

judge_chain = judge_prompt | llm | StrOutputParser()

def evaluate_retrieval(question: str, retriever, n_chunks: int = 4, label: str = "") -> dict:
    """Evaluate the quality of retrieved chunks for a given question using LLM-as-a-Judge."""
    retrieved = retriever.invoke(question)[:n_chunks]
    scores = []
    
    print(f"\n{'='*60}")
    print(f"EVALUATION: {label}")
    print(f"Question: {question[:70]}...")
    print(f"{'='*60}")
    
    for i, doc in enumerate(retrieved):
        result_str = judge_chain.invoke({
            "question": question,
            "chunk": doc.page_content[:600]
        })
        try:
            start = result_str.find("{")
            end = result_str.rfind("}") + 1
            result = json.loads(result_str[start:end])
        except:
            result = {"score": 3, "reason": "Parse error — defaulting to 3"}
        
        title = doc.metadata.get("source_title", "Unknown")[:40]
        page = doc.metadata.get("page", 0)
        scores.append(result["score"])
        
        bar = "█" * result["score"] + "░" * (5 - result["score"])
        print(f"  Chunk {i+1} [{bar}] {result['score']}/5 | {title}... (p.{page})")
        print(f"         Reason: {result['reason']}")
    
    avg_score = sum(scores) / len(scores) if scores else 0
    print(f"\n  📊 Average Relevance Score: {avg_score:.1f}/5.0")
    return {"scores": scores, "average": avg_score, "label": label}

# ── Evaluation 1: On-topic question ───────────────────────────────────────────
eval1 = evaluate_retrieval(
    question="What planning mechanisms do agentic LLM systems use for multi-step reasoning?",
    retriever=arxiv_retriever,
    n_chunks=4,
    label="ON-TOPIC QUESTION"
)



EVALUATION: ON-TOPIC QUESTION
Question: What planning mechanisms do agentic LLM systems use for multi-step rea...
  Chunk 1 [█████] 5/5 | paper 01 Agentic Reasoning for Large Lan... (p.47)
         Reason: The chunk directly describes planning mechanisms in agentic LLM systems, including decomposition of goals, tool selection, plan revision, and specific examples like ProtAgents and Eunomia.
  Chunk 2 [███░░] 3/5 | paper 03 When Reasoning Beats Scale  A 1... (p.0)
         Reason: The chunk mentions planning in LLM systems and agentic frameworks but does not specify the mechanisms used for multi-step reasoning.
  Chunk 3 [████░] 4/5 | paper 01 Agentic Reasoning for Large Lan... (p.1)
         Reason: The chunk discusses planning mechanisms like Chain-of-Thought prompting, decomposition, and program-aided solving, which are relevant to multi-step reasoning in agentic LLM systems, but does not provide a comprehensive or direct answer.
  Chunk 4 [████░] 4/5 | paper 01 Agentic Reasoning f

In [16]:
# ── Evaluation 2: Off-topic question ─────────────────────────────────────────
eval2 = evaluate_retrieval(
    question="What is the best recipe for making sourdough bread at home?",
    retriever=arxiv_retriever,
    n_chunks=4,
    label="OFF-TOPIC QUESTION"
)

# ── Evaluation 3: Partially relevant question ─────────────────────────────────
eval3 = evaluate_retrieval(
    question="How do transformer attention mechanisms work in general?",
    retriever=arxiv_retriever,
    n_chunks=4,
    label="PARTIALLY RELEVANT QUESTION"
)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("RETRIEVAL QUALITY SUMMARY")
print("="*60)
for ev in [eval1, eval2, eval3]:
    bar_val = int(ev["average"])
    bar = "█" * bar_val + "░" * (5 - bar_val)
    print(f"  {ev['label']:<35} [{bar}] {ev['average']:.1f}/5.0")
print()
print("💡 A well-functioning RAG system shows a clear gap between")
print("   on-topic and off-topic retrieval scores.")
print("   High on-topic scores → retriever is working correctly.")
print("   Low off-topic scores → model will correctly say 'I don't know'.")



EVALUATION: OFF-TOPIC QUESTION
Question: What is the best recipe for making sourdough bread at home?...
  Chunk 1 [█░░░░] 1/5 | paper 01 Agentic Reasoning for Large Lan... (p.27)
         Reason: The chunk discusses software tools and APIs, which are unrelated to sourdough bread recipes.
  Chunk 2 [█░░░░] 1/5 | paper 01 Agentic Reasoning for Large Lan... (p.65)
         Reason: The chunk does not mention sourdough bread or recipes and is unrelated to the question.
  Chunk 3 [█░░░░] 1/5 | paper 01 Agentic Reasoning for Large Lan... (p.58)
         Reason: The chunk discusses reinforcement learning and research efficiency, which is unrelated to sourdough bread recipes.
  Chunk 4 [█░░░░] 1/5 | paper 01 Agentic Reasoning for Large Lan... (p.99)
         Reason: The chunk discusses academic papers on language models and does not mention sourdough bread or recipes.

  📊 Average Relevance Score: 1.0/5.0

EVALUATION: PARTIALLY RELEVANT QUESTION
Question: How do transformer attention mechanism

---
## 📚 Summary: What We Built

| Component | Technology | Purpose |
|-----------|-----------|---------|
| LLM | Azure OpenAI GPT-4.1 | Text generation and reasoning |
| Embeddings | Azure text-embedding (1536-dim) | Semantic vector representation |
| Vector Store | FAISS | Fast similarity search |
| Document Loader | WikipediaLoader, PyPDFLoader | Ingesting raw documents |
| Text Splitter | RecursiveCharacterTextSplitter | Chunking documents |
| Chain Builder | LangChain LCEL (`|` operator) | Composing the pipeline |
| Paper Source | arXiv API + requests | Real scientific papers |
| Evaluation | LLM-as-a-Judge | Measuring retrieval quality |

---

## 🔑 Key Takeaways

**RAG solves three fundamental problems with pure LLMs:**

1. **Hallucination** — By grounding answers in retrieved documents, the model cannot fabricate facts that aren't in the context.
2. **Knowledge cutoff** — The vector database can be updated with new documents at any time, without retraining the model.
3. **Traceability** — With proper prompt engineering, every claim can be traced to a specific source, page, and author.

**Design decisions that matter:**

- **Chunk size and overlap** directly impact retrieval quality. Larger chunks provide more context but may dilute relevance; smaller chunks are more precise but may lose context.
- **The number of retrieved chunks (k)** is a trade-off between coverage and noise. More chunks = more context but also more irrelevant text.
- **Prompt engineering for citations** is as important as the retrieval mechanism itself. A well-designed prompt prevents the model from mixing its parametric knowledge with retrieved context.
- **Evaluating retrieval quality** using LLM-as-a-Judge is essential before deploying a RAG system in production.

---

*Developed for educational purposes in advanced AI systems design.*
